In [32]:
# Raw layer has 3 CSV files, which we are reading into Spark dataframes. 
# And then these dataframes are written to Bronze layer in Parquet file format

In [67]:
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("BronzeLayerIngestion").getOrCreate()

customers_s = spark.read.csv("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/raw/olist_customers_dataset.csv", header = True, inferSchema = True)
orders_s = spark.read.csv("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/raw/olist_orders_dataset.csv", header = True, inferSchema = True)
order_items_s = spark.read.csv("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/raw/olist_order_items_dataset.csv", header = True, inferSchema = True)
payments_s = spark.read.csv("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/raw/olist_order_payments_dataset.csv", header = True, inferSchema = True)


In [68]:
def add_metadata_bronze(df, table_name):
    df = df.withColumn("ingestion_timestamp", current_timestamp()).withColumn("source_system", lit("olist_ecommerce"))
    print(f"\nMetadata Ingested for {table_name}")

    Total_rows = df.count()
    print(f"Total no. of rows for {table_name}: {Total_rows}")
    return df

customers_b = add_metadata_bronze(customers_s, "customers")
orders_b = add_metadata_bronze(orders_s, "orders")
order_items_b = add_metadata_bronze(order_items_s, "order_items")
payments_b = add_metadata_bronze(payments_s, "payments")

print("\nBronze Ingestion Layer completed successfully!!")



Metadata Ingested for customers
Total no. of rows for customers: 99441

Metadata Ingested for orders
Total no. of rows for orders: 99441

Metadata Ingested for order_items
Total no. of rows for order_items: 112650

Metadata Ingested for payments
Total no. of rows for payments: 103886

Bronze Ingestion Layer completed successfully!!


In [73]:
customers_null_count = customers_b.filter(col("customer_id").isNull()).count()
print(f"Count of Nulls in customers table : {customers_null_count}")

customers_duplicate_count = customers_b.groupBy("customer_id").count().filter("count>1").count()
print(f"Count of Duplicates in customers table : {customers_duplicate_count}")

Count of Nulls in customers table : 0
Count of Duplicates in customers table : 0


In [74]:
orders_null_count = orders_b.filter(col("order_id").isNull()).count()
print(f"Count of Nulls in Orders table : {orders_null_count}")

orders_duplicate_count = orders_b.groupBy("order_id").count().filter("count>1").count()
print(f"Count of Duplicates in Orders table : {orders_duplicate_count}")

Count of Nulls in Orders table : 0
Count of Duplicates in Orders table : 0


In [75]:
customers_b.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/bronze/olist_customers_dataset")
orders_b.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/bronze/olist_orders_dataset")
order_items_b.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/bronze/olist_order_items_dataset")
payments_b.write.mode("overwrite").parquet("/Users/shamilinemmani/Desktop/ecommerce_pipeline/data/bronze/olist_order_payments_dataset")